# Model Context Protocol (MCP) - Server

This example demonstrates how to integrate agent with external tools using the Model Context Protocol (MCP). It shows how to create a simple MCP server that provides calculator functionality and connect a agent to use these tools.

## Overview

| Feature            | Description                            |
| ------------------ | -------------------------------------- |
| **Tool Used**      | MCPAgentTool                           |
| **Protocol**       | Model Context Protocol (MCP)           |
| **Complexity**     | Intermediate                           |
| **Agent Type**     | Single Agent                           |
| **Interaction**    | Command Line Interface                 |

More details on MCP can be found at:
- https://modelcontextprotocol.io
- https://github.com/microsoft/mcp
- https://github.com/awslabs/mcp
- https://github.com/google/mcp
- https://github.com/oracle/mcp

## Tool Overview

The Model Context Protocol (MCP) enables agent to use tools provided by external servers, connecting conversational AI with specialized functionality. The SDK provides the `MCPAgentTool` class which adapts MCP tools to the agent framework's tool interface. 
The `MCPAgentTool` is loaded via an MCPClient, which represents a connection to an external server that provides tools for the agent to use.



In [1]:
%%time
import warnings
warnings.filterwarnings("ignore")

!python3 -m pip install --upgrade pip
!python3 -m pip3 install --upgrade pip

/usr/local/bin/python3: No module named pip3
CPU times: user 7.91 ms, sys: 14.4 ms, total: 22.3 ms
Wall time: 972 ms


## Installation

Install required packages for MCP server and agent integration.

- langchain
    - https://academy.langchain.com
    - https://pypi.org/project/langchain/
- strands-agents
    - https://strandsagents.com/latest/
    - https://pypi.org/project/strands-agents/

In [2]:
%pip install -U -q mcp langchain langchain-core strands-agents strands-agents-tools ipython-autotime --use-deprecated=legacy-resolver

%load_ext autotime

Note: you may need to restart the kernel to use updated packages.
time: 135 μs (started: 2025-12-08 10:27:05 -08:00)


## Create MCP Calculator Server

The MCP server provides calculator tools that can be used by agents.


In [3]:
# Create MCP server instance
from typing import Any, Sequence
from mcp.server import Server
from mcp.types import Tool, TextContent

calculator_server = Server("calculator-server")


# Store handler functions for direct access
async def list_tools_handler() -> list[Tool]:
    """List available calculator tools."""
    return [
        Tool(
            name="add",
            description="Add two numbers together",
            inputSchema={
                "type": "object",
                "properties": {
                    "a": {"type": "number", "description": "First number"},
                    "b": {"type": "number", "description": "Second number"},
                },
                "required": ["a", "b"],
            },
        ),
        Tool(
            name="subtract",
            description="Subtract the second number from the first number",
            inputSchema={
                "type": "object",
                "properties": {
                    "a": {"type": "number", "description": "First number"},
                    "b": {"type": "number", "description": "Second number"},
                },
                "required": ["a", "b"],
            },
        ),
        Tool(
            name="multiply",
            description="Multiply two numbers together",
            inputSchema={
                "type": "object",
                "properties": {
                    "a": {"type": "number", "description": "First number"},
                    "b": {"type": "number", "description": "Second number"},
                },
                "required": ["a", "b"],
            },
        ),
        Tool(
            name="divide",
            description="Divide the first number by the second number",
            inputSchema={
                "type": "object",
                "properties": {
                    "a": {"type": "number", "description": "First number"},
                    "b": {"type": "number", "description": "Second number"},
                },
                "required": ["a", "b"],
            },
        ),
        Tool(
            name="power",
            description="Raise the first number to the power of the second number",
            inputSchema={
                "type": "object",
                "properties": {
                    "a": {"type": "number", "description": "Base number"},
                    "b": {"type": "number", "description": "Exponent"},
                },
                "required": ["a", "b"],
            },
        ),
    ]


async def call_tool_handler(
    name: str, arguments: dict[str, Any]
) -> Sequence[TextContent]:
    """Execute calculator tool operations."""
    if name == "add":
        result = arguments["a"] + arguments["b"]
        return [TextContent(type="text", text=str(result))]
    elif name == "subtract":
        result = arguments["a"] - arguments["b"]
        return [TextContent(type="text", text=str(result))]
    elif name == "multiply":
        result = arguments["a"] * arguments["b"]
        return [TextContent(type="text", text=str(result))]
    elif name == "divide":
        if arguments["b"] == 0:
            return [TextContent(type="text", text="Error: Division by zero")]
        result = arguments["a"] / arguments["b"]
        return [TextContent(type="text", text=str(result))]
    elif name == "power":
        result = arguments["a"] ** arguments["b"]
        return [TextContent(type="text", text=str(result))]
    else:
        return [TextContent(type="text", text=f"Unknown tool: {name}")]


# Register handlers with the server
calculator_server.list_tools()(list_tools_handler)
calculator_server.call_tool()(call_tool_handler)

print("Calculator MCP server defined successfully!")

Calculator MCP server defined successfully!
time: 260 ms (started: 2025-12-08 10:27:05 -08:00)


## Create In-Process MCP Client

For notebook environments, we'll create an in-process MCP client.


In [4]:
class InProcessMCPClient:
    """In-process MCP client for notebook environments."""

    def __init__(self, server, list_tools_handler, call_tool_handler):
        self.server = server
        self.list_tools_handler = list_tools_handler
        self.call_tool_handler = call_tool_handler
        self._tools_cache = None

    async def list_tools(self):
        """List available tools from the server."""
        if self._tools_cache is None:
            tools = await self.list_tools_handler()
            self._tools_cache = {tool.name: tool for tool in tools}
        return self._tools_cache

    async def call_tool(self, name: str, arguments: dict):
        """Call a tool on the server."""
        return await self.call_tool_handler(name, arguments)


# Create in-process client
mcp_client = InProcessMCPClient(
    calculator_server, list_tools_handler, call_tool_handler
)
print("In-process MCP client created successfully!")

mcp_client

In-process MCP client created successfully!


time: 2.83 ms (started: 2025-12-08 10:27:05 -08:00)


## Test MCP Server Tools

Test the calculator tools directly.


In [5]:
async def test_calculator_tools():
    """Test all calculator tools."""
    print("Testing calculator tools...\n")

    result = await call_tool_handler("add", {"a": 10, "b": 5})
    print(f"Add: 10 + 5 = {result[0].text}")

    result = await call_tool_handler("subtract", {"a": 10, "b": 5})
    print(f"Subtract: 10 - 5 = {result[0].text}")

    result = await call_tool_handler("multiply", {"a": 10, "b": 5})
    print(f"Multiply: 10 * 5 = {result[0].text}")

    result = await call_tool_handler("divide", {"a": 10, "b": 5})
    print(f"Divide: 10 / 5 = {result[0].text}")

    result = await call_tool_handler("power", {"a": 2, "b": 8})
    print(f"Power: 2 ^ 8 = {result[0].text}")


await test_calculator_tools()

Testing calculator tools...

Add: 10 + 5 = 15
Subtract: 10 - 5 = 5
Multiply: 10 * 5 = 50
Divide: 10 / 5 = 2.0
Power: 2 ^ 8 = 256
time: 723 μs (started: 2025-12-08 10:27:05 -08:00)
